<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


**Method: Random Forest classifier**

 Each page is labeled True or False (for whether it is declining) based on its  change in impressions (±20% threshold). The task is therefore classification.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Current split:** Stratified random split, 80% train / 20% test, stratified on the target label with a fixed random seed.

**Next step:** This is a baseline split only. In the next assignment (validation audit), we will switch to a client-grouped split using `GroupShuffleSplit`.

##### Load data: features and `is_declining` label

In [1]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login


TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"


FEAT_START, FEAT_END   = "2026-01-01", "2026-02-28"
MID                    = "2026-01-31"
LABEL_START, LABEL_END = "2026-03-01", "2026-04-30"
MIN_IMPR = 100


con.execute(f"""
CREATE OR REPLACE TEMP TABLE content_slice AS
SELECT d.client_hash_id, d.content_hash_id
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
WHERE d.month IN ('2026-01','2026-02')
  AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
  AND CAST(c.gsc_data_start AS DATE) < DATE '{FEAT_START}'
GROUP BY d.client_hash_id, d.content_hash_id
HAVING SUM(d.gsc_impressions) >= {MIN_IMPR}
""")
print("pages:", con.execute("SELECT COUNT(*) FROM content_slice").fetchone()[0])


con.execute(f"""
CREATE OR REPLACE TEMP TABLE dataset AS
WITH daily AS (
    SELECT d.client_hash_id, d.content_hash_id, d.report_date,
           d.gsc_impressions, d.gsc_clicks, d.gsc_sum_position,
           d.ga4_pageviews, d.ga4_sessions, d.ga4_users,
           d.ga4_engaged_sessions, d.ga4_total_engagement_sec,
           COALESCE(d.report_date >= CAST(c.ga4_data_start AS DATE), FALSE) AS ga4_ok
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN content_slice s
      ON s.client_hash_id = d.client_hash_id AND s.content_hash_id = d.content_hash_id
    JOIN read_parquet('{SRC_CLIENTS}') c ON c.client_hash_id = d.client_hash_id
    WHERE d.month IN ('2026-01','2026-02','2026-03','2026-04')
      AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{LABEL_END}'
      AND d.report_date >= CAST(c.gsc_data_start AS DATE)
),
agg AS (
    SELECT client_hash_id, content_hash_id,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                         AS f01,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)                 AS f02,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)           AS f05,
      SUM(CASE WHEN ga4_ok THEN ga4_engaged_sessions END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f06,
      SUM(CASE WHEN ga4_ok THEN ga4_total_engagement_sec END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f07,
      SUM(CASE WHEN ga4_ok THEN ga4_pageviews END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f08,
      SUM(CASE WHEN ga4_ok THEN ga4_users END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f09
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2
),
halves AS (
    SELECT client_hash_id, content_hash_id,
      (report_date > DATE '{MID}') AS is_h2,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                AS avg_impr,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)        AS ctr,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)  AS pos
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2, 3
    HAVING SUM(gsc_impressions) > 0
),
trend AS (
    SELECT client_hash_id, content_hash_id,
      (MAX(CASE WHEN is_h2 THEN avg_impr END) - MAX(CASE WHEN NOT is_h2 THEN avg_impr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN avg_impr END), 0)       AS f03,
      (MAX(CASE WHEN is_h2 THEN ctr END) - MAX(CASE WHEN NOT is_h2 THEN ctr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN ctr END), 0)            AS f04,
      MAX(CASE WHEN is_h2 THEN pos END) - MAX(CASE WHEN NOT is_h2 THEN pos END) AS f14
    FROM halves
    GROUP BY 1, 2
),
dim AS (
    SELECT content_hash_id,
      DATE_DIFF('day',
        CAST(CASE WHEN content_updated_date IS NOT NULL
                   AND content_updated_date <= DATE '{FEAT_END}'
                  THEN content_updated_date ELSE content_created_date END AS DATE),
        DATE '{FEAT_END}')                                                   AS f10,
      CASE WHEN keyword_hash_id IS NULL OR keyword_hash_id = '' THEN NULL
           ELSE COUNT(*) OVER (PARTITION BY client_hash_id, keyword_hash_id) - 1
      END                                                                    AS f11,
      CASE WHEN main_intent IS NULL THEN NULL
           ELSE DENSE_RANK() OVER (ORDER BY main_intent) - 1 END             AS f12,
      backlinks                                                              AS f13
    FROM read_parquet('{SRC_CONTENT}')
),
lab AS (
    SELECT client_hash_id, content_hash_id,
      SUM(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN report_date END), 0)                                     AS avg_base,
      COALESCE(SUM(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN report_date END), 0), 0)                                 AS avg_next
    FROM daily
    GROUP BY 1, 2
)
SELECT s.content_hash_id AS content_id,
       s.client_hash_id  AS client_id,
       a.f01 AS avg_daily_impressions,
       a.f02 AS ctr,
       t.f03 AS impressions_trend,
       t.f04 AS ctr_trend,
       a.f05 AS weighted_avg_position,
       a.f06 AS engagement_rate,
       a.f07 AS engagement_time_per_session,
       a.f08 AS avg_daily_pageviews,
       a.f09 AS avg_daily_users,
       m.f10 AS page_age_days,
       m.f11 AS same_keyword_pages,
       m.f12 AS main_intent_code,
       m.f13 AS backlinks,
       t.f14 AS position_trend,
       CASE WHEN l.avg_next < 0.8 * l.avg_base THEN TRUE
            ELSE FALSE END AS is_declining
FROM content_slice s
LEFT JOIN agg   a ON a.client_hash_id = s.client_hash_id AND a.content_hash_id = s.content_hash_id
LEFT JOIN trend t ON t.client_hash_id = s.client_hash_id AND t.content_hash_id = s.content_hash_id
LEFT JOIN lab   l ON l.client_hash_id = s.client_hash_id AND l.content_hash_id = s.content_hash_id
LEFT JOIN dim   m ON m.content_hash_id = s.content_hash_id
""")

df = con.execute("SELECT * FROM dataset").df()
print(df.shape)
df

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 82612


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(82612, 17)


,content_id,client_id,avg_daily_impressions,ctr,impressions_trend,ctr_trend,weighted_avg_position,engagement_rate,engagement_time_per_session,avg_daily_pageviews,avg_daily_users,page_age_days,same_keyword_pages,main_intent_code,backlinks,position_trend,is_declining
0,content_2beea676d1f82ac0,client_73cda7b4e4f265ea,3.237288,0.000000,1.737662,NaN,10.753927,NaN,NaN,NaN,NaN,3,0,0,0,-4.099198,False
1,content_6692551c97224ae7,client_ff644d8251367cbb,8.627119,0.003929,0.408642,NaN,10.170923,0.0,1.0,0.067797,0.067797,393,0,0,<NA>,-2.764364,False
2,content_37447482347acea2,client_ff644d8251367cbb,2.016949,0.000000,2.960165,NaN,10.789916,0.0,0.0,0.016949,0.016949,3,0,0,<NA>,-6.174524,False
3,content_d1e74562ab1359b3,client_ff644d8251367cbb,3.576271,0.000000,3.454932,NaN,14.246445,NaN,NaN,0.000000,0.000000,359,0,0,<NA>,5.539589,False
4,content_8e96590ddf859673,client_ff644d8251367cbb,26.915254,0.001259,1.723234,-1.000000,6.899874,0.0,0.0,0.067797,0.067797,401,0,0,<NA>,-0.346147,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82607,content_0a3dbc35519aaf88,client_d211cb07b9059bab,13.711864,0.000000,0.120913,NaN,9.386897,NaN,NaN,0.000000,0.000000,3,<NA>,<NA>,<NA>,0.007579,False
82608,content_98154a3b5baab58e,client_d211cb07b9059bab,3.338983,0.010152,NaN,NaN,7.664975,NaN,NaN,0.000000,0.000000,173,<NA>,<NA>,<NA>,NaN,False
82609,content_81028f7bce06c3ef,client_d211cb07b9059bab,12.932203,0.001311,0.015345,-1.000000,10.483617,NaN,NaN,0.000000,0.000000,3,<NA>,<NA>,<NA>,0.354423,False
82610,content_92b90353e195ddb4,client_d211cb07b9059bab,2.440678,0.013889,0.345604,NaN,8.944444,0.0,0.0,0.200000,0.100000,3,<NA>,<NA>,<NA>,-1.082765,True


Train test split

In [2]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["is_declining"],
    random_state=SEED,
)
train_df, test_df = train_df.copy(), test_df.copy()

print("seed:", SEED)
print("train:", len(train_df), "pages |", train_df["client_id"].nunique(), "clients")
print("test: ", len(test_df), "pages |", test_df["client_id"].nunique(), "clients")
print("\ntrain label share:\n", train_df["is_declining"].value_counts(normalize=True).round(3))
print("\ntest label share:\n", test_df["is_declining"].value_counts(normalize=True).round(3))


shared = set(train_df["client_id"]) & set(test_df["client_id"])
print("\nclients in both train and test:", len(shared))

seed: 42
train: 66089 pages | 28 clients
test:  16523 pages | 26 clients

train label share:
 is_declining
False    0.666
True     0.334
Name: proportion, dtype: float64

test label share:
 is_declining
False    0.666
True     0.334
Name: proportion, dtype: float64

clients in both train and test: 26


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

FEATURES = [
    "avg_daily_impressions", "ctr", "impressions_trend", "ctr_trend",
    "weighted_avg_position", "engagement_rate", "engagement_time_per_session",
    "avg_daily_pageviews", "avg_daily_users", "page_age_days",
    "same_keyword_pages", "main_intent_code", "backlinks", "position_trend",
]



X_train, y_train = train_df[FEATURES], train_df["is_declining"]
X_test,  y_test  = test_df[FEATURES],  test_df["is_declining"]

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, pred), 4))
from sklearn.metrics import f1_score
print("macro F1:", round(f1_score(y_test, pred, average="macro"), 4))

test accuracy: 0.7295
macro F1: 0.7077


In [7]:
from google.colab import drive
drive.mount("/content/drive")

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score


RULE_PATH = "/content/drive/MyDrive/my_dataframe.csv"
ID_COL    = "content"
SCORE_COL = "score"
K_LIST    = [50,100, 200 , 500]


rule = pd.read_csv(RULE_PATH)
rule = (rule[[ID_COL, SCORE_COL]]
        .rename(columns={ID_COL: "content_id", SCORE_COL: "rule_score"})
        .drop_duplicates("content_id"))


t = test_df[["content_id", "client_id", "is_declining"]].copy()
t["model_score"] = rf.predict_proba(X_test)[:, list(rf.classes_).index(True)]
t = t.merge(rule, on="content_id", how="left")



c = t.dropna(subset=["rule_score"]).reset_index(drop=True)


def report(d, col, name):
    y = d["is_declining"].astype(int).to_numpy()
    s = d[col].to_numpy()
    order = np.argsort(-s, kind="stable")
    row = {"method": name,
           "roc_auc": round(roc_auc_score(y, s), 4),
           "avg_precision": round(average_precision_score(y, s), 4)}
    for k in K_LIST:
        row[f"precision@{k}"] = round(y[order[:k]].mean(), 3)
    return row

res = pd.DataFrame([report(c, "rule_score", "rule"),
                    report(c, "model_score", "random_forest")])

display(res.set_index("method").round(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,roc_auc,avg_precision,precision@50,precision@100,precision@200,precision@500
method,,,,,,
rule,0.550,0.360,0.78,0.60,0.48,0.316
random_forest,0.806,0.694,0.98,0.98,0.96,0.936


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

 Model performance per client

In [5]:
import pandas as pd


t = test_df[["client_id", "is_declining"]].copy()
t["pred"] = rf.predict_proba(X_test)[:, list(rf.classes_).index(True)] > 0.5
t["group"] = "TN"
t.loc[t["pred"] & ~t["is_declining"], "group"] = "FP"
t.loc[~t["pred"] & t["is_declining"], "group"] = "FN"
t.loc[t["pred"] & t["is_declining"], "group"] = "TP"
print(t["group"].value_counts())


g = t.groupby("client_id").agg(
    pages=("group", "size"),
    fp=("group", lambda s: (s == "FP").sum()),
    fn=("group", lambda s: (s == "FN").sum()),
)
g["errors"] = g["fp"] + g["fn"]
g["error_rate"] = g["errors"] / g["pages"]
g["share_of_errors"] = g["errors"] / g["errors"].sum()
g["share_of_pages"] = g["pages"] / g["pages"].sum()
display(g.sort_values("errors", ascending=False).head(10).round(3))


group
TN    8286
TP    3768
FP    2720
FN    1749
Name: count, dtype: int64


,pages,fp,fn,errors,error_rate,share_of_errors,share_of_pages
client_id,,,,,,,
client_62f4a7e64f5e0096,3408,751,458,1209,0.355,0.271,0.206
client_73cda7b4e4f265ea,3948,758,353,1111,0.281,0.249,0.239
client_e547b89c05043229,1760,263,154,417,0.237,0.093,0.107
client_fef1a8f436438636,1384,214,129,343,0.248,0.077,0.084
client_23a62021009f63c4,2110,109,232,341,0.162,0.076,0.128
client_08a6a72ff48e62c0,1134,173,160,333,0.294,0.075,0.069
client_9958f0a7ae1df715,414,103,64,167,0.403,0.037,0.025
client_400c21c81c8b46ef,303,110,6,116,0.383,0.026,0.018
client_3197e6291363b4db,263,55,54,109,0.414,0.024,0.016


Feature importance

Importance is computed two ways:

- Impurity-based: the built-in `feature_importances_` from the Random Forest.
- Permutation-based: shuffle one feature column at a time and measure how much the score drops.

In [6]:
from sklearn.inspection import permutation_importance

imp = pd.DataFrame({
    "built_in": rf.feature_importances_,
}, index=FEATURES)

perm = permutation_importance(rf, X_test, y_test, scoring="f1_macro",
                              n_repeats=10, random_state=42, n_jobs=-1)
imp["permutation"] = perm.importances_mean

display(imp.sort_values("permutation", ascending=False).round(4))

,built_in,permutation
impressions_trend,0.3380,0.1018
page_age_days,0.1790,0.0644
ctr,0.0761,0.0205
weighted_avg_position,0.0782,0.0144
main_intent_code,0.0386,0.0092
backlinks,0.0231,0.0088
position_trend,0.0790,0.0069
avg_daily_impressions,0.0490,0.0055
avg_daily_pageviews,0.0358,0.0005
avg_daily_users,0.0244,0.0004


**Where the model is wrong:** errors concentrate in certain clients (most importantly `client_62f4`, with a 35% error rate against 27% overall).

**What it leans on:** the two strongest features are `impressions_trend` and `page_age_days`.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.